In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
torch.manual_seed(1337)

Using device: cuda


In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text)}")
print(text[:500])

--2026-09-20 22:43:13--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-09-20 22:43:13 (226 MB/s) - ‘input.txt’ saved [1115394/1115394]

Length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdi

In [3]:
# Find all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary: {''.join(chars)}")
print(f"Vocab size: {vocab_size}")

# Create mapping from characters to integers and back
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# Quick test
test = encode("Hello there")
print(test)
print(decode(test))

Vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65
[20, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43]
Hello there


In [4]:
# Encode the entire dataset into a tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}, dtype: {data.dtype}")

# Train/val split (90/10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# Hyperparameters for batching
block_size = 8   # how many characters of context the model sees at once
batch_size = 4    # how many independent sequences we process in parallel

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

Data shape: torch.Size([1115394]), dtype: torch.int64
inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')


In [5]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C) = (batch, time, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]  # take only the last time step's predictions
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)

logits, loss = m(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")

# Generate from untrained model (should be gibberish)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))

Logits shape: torch.Size([32, 65])
Loss: 5.0364

yq$;tfBfROkNdcuwdZZTkOMl;,ertK
w:!PLCkMBbeA$3:XaSGJO-3p&M-c?KL3auhpFYVXJFhNNNuhq$OMxv.tbVFYdXlrFZaAe


In [6]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32  # bump this up now that we're actually training
max_iters = 5000
eval_interval = 500

@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")

step 0: train loss 4.6355, val loss 4.6491
step 500: train loss 4.1103, val loss 4.1273
step 1000: train loss 3.6894, val loss 3.7089
step 1500: train loss 3.3689, val loss 3.3832
step 2000: train loss 3.1215, val loss 3.1397
step 2500: train loss 2.9432, val loss 2.9460
step 3000: train loss 2.8016, val loss 2.8234
step 3500: train loss 2.7148, val loss 2.7263
step 4000: train loss 2.6480, val loss 2.6475
step 4500: train loss 2.5993, val loss 2.6146

Final loss: 2.5560


In [7]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=300)[0].tolist()))


Wawice my.

HDEdacomzy mug
Yow&$LMETfuisth ble mil;KI ll, ath iree sengmin lat HNGEdrovDEs, and Win nghir.
TWjus!
el lind me l.
lishe ce hiry ptug; aisspllw y.
Hllin's n Bfopetelives
MPOFGll, d mothakleo Windo whthCorib3MI'Tham dourive we hixend t so mower; te

ANk d nterurt f s ar igr Wam:

Thiny a


In [8]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32  # batch, time (sequence length), channels (embedding dim)
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, head_size)
q = query(x)  # (B, T, head_size)
wei = q @ k.transpose(-2, -1)  # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
wei = wei * head_size**-0.5     # scale, to keep variance stable

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  # prevent attending to future tokens
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

print(f"wei shape: {wei.shape}")
print(wei[0])  # attention pattern for the first sequence in the batch
print(f"out shape: {out.shape}")

wei shape: torch.Size([4, 8, 8])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)
out shape: torch.Size([4, 8, 16])


In [9]:
# Hyperparameters for our small model
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
block_size = 32  # let's also bump context length up a bit from 8

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ simple MLP: gives the model room to 'think' after gathering info via attention """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ one transformer block: communication (attention) then computation (feedforward) """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # residual connection around attention
        x = x + self.ffwd(self.ln2(x))  # residual connection around feedforward
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]  # crop to last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = GPTLanguageModel()
m = model.to(device)
print(f"Model has {sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")

Model has 0.21M parameters


In [10]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
max_iters = 5000
eval_interval = 500

@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")

step 0: train loss 4.3294, val loss 4.3273
step 500: train loss 2.1824, val loss 2.2018
step 1000: train loss 1.9617, val loss 2.0330
step 1500: train loss 1.8364, val loss 1.9623
step 2000: train loss 1.7415, val loss 1.8845
step 2500: train loss 1.6936, val loss 1.8519
step 3000: train loss 1.6609, val loss 1.8021
step 3500: train loss 1.6267, val loss 1.7931
step 4000: train loss 1.6089, val loss 1.7800
step 4500: train loss 1.5827, val loss 1.7670

Final loss: 1.5797


In [11]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


And they bride.

NORTHUMBERLAND:
Are my be die.
Sher-day we galants: 'tis he use,
The tracterlassanes with my fafts' come!

WARY:
Good of it her! All heling, and is ender contlain:
Givelve the death prepirecerable. And no me little;
Honce, you can answer why.

KING HIRGBRO:
Yet the hopest would that
morn-cound dramisbalt's them sails
And him he poor of his burder hinders,
If so; sir. Where threy all of which Prince, Some,
Had some sticking:
Ard all his graced of the kings tey it-lisments after y


In [12]:
def generate_with_sampling(model, idx, max_new_tokens, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature  # scale by temperature

        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')  # mask everything outside top-k

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("--- temperature=0.7, top_k=10 (conservative) ---")
print(decode(generate_with_sampling(m, context, 300, temperature=0.7, top_k=10)[0].tolist()))

print("\n--- temperature=1.3, top_k=None (wild) ---")
print(decode(generate_with_sampling(m, context, 300, temperature=1.3, top_k=None)[0].tolist()))

--- temperature=0.7, top_k=10 (conservative) ---

Men your greate.

CLARENCE:
Them so a changed-senter.

LEONTES:
The show she hath a lieging her must have my should so,
I making he must of thou done, be silves, if her commans in mercy some hath daughter and the mystate of my hunders,
And he condemn may shall be a long
That but with soul have may m

--- temperature=1.3, top_k=None (wild) ---

'Tis stiflly no'er in, much him;
Nhile. I know the nape, do thing
severideous is ofter nor the summon apprin on what than the ruil perclaimb marry so hate!

SLY.
Thwatch time wout if form;
you contome, I take it we have it, Quw nike, Much nurchai h,
Ther Lawsadome, withal'd orn against, merning nose


In [13]:
def get_stats(ids):
    """Count frequency of adjacent pairs"""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    """Replace all occurrences of `pair` in `ids` with `new_id`"""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

# Start from raw bytes of the text (0-255)
tokens = list(text.encode('utf-8'))
print(f"Starting tokens (bytes): {len(tokens)}")

num_merges = 300  # how many new tokens to create beyond the base 256 bytes
merges = {}  # (pair) -> new_id
ids = list(tokens)

for i in range(num_merges):
    stats = get_stats(ids)
    if not stats:
        break
    pair = max(stats, key=stats.get)  # most frequent pair
    new_id = 256 + i
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
    if i % 50 == 0:
        print(f"merge {i}: {pair} -> {new_id} (had {stats[pair]} occurrences)")

print(f"\nFinal token count: {len(ids)} (compression ratio: {len(tokens)/len(ids):.2f}x)")
bpe_vocab_size = 256 + num_merges
print(f"New vocab size: {bpe_vocab_size}")

Starting tokens (bytes): 1115394
merge 0: (101, 32) -> 256 (had 27643 occurrences)
merge 50: (108, 105) -> 306 (had 2358 occurrences)
merge 100: (119, 303) -> 356 (had 1258 occurrences)
merge 150: (97, 265) -> 406 (had 849 occurrences)
merge 200: (115, 276) -> 456 (had 636 occurrences)
merge 250: (280, 259) -> 506 (had 486 occurrences)

Final token count: 549002 (compression ratio: 2.03x)
New vocab size: 556


In [14]:
def bpe_encode(s):
    tokens = list(s.encode('utf-8'))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        # find the pair with the lowest merge index (earliest/most important merge)
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))
        if pair not in merges:
            break  # no more applicable merges
        tokens = merge(tokens, pair, merges[pair])
    return tokens

def bpe_decode(ids):
    # build reverse vocab: id -> bytes
    vocab = {idx: bytes([idx]) for idx in range(256)}
    for (p0, p1), idx in merges.items():
        vocab[idx] = vocab[p0] + vocab[p1]
    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode('utf-8', errors='replace')

# Sanity check: round trip
test = "Hello there, First Citizen!"
encoded = bpe_encode(test)
decoded = bpe_decode(encoded)
print(f"Original: {test}")
print(f"Encoded ({len(encoded)} tokens): {encoded}")
print(f"Decoded: {decoded}")
print(f"Match: {test == decoded}")

Original: Hello there, First Citizen!
Encoded (14 tokens): [72, 389, 269, 368, 302, 70, 299, 310, 67, 316, 105, 122, 270, 33]
Decoded: Hello there, First Citizen!
Match: True


In [15]:
# Re-encode the entire dataset with BPE
bpe_data = torch.tensor(bpe_encode(text), dtype=torch.long)
print(f"BPE data shape: {bpe_data.shape}")

n = int(0.9 * len(bpe_data))
train_data = bpe_data[:n]
val_data = bpe_data[n:]

vocab_size = bpe_vocab_size  # 556, instead of 65
block_size = 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Rebuild the model with the new vocab size
model = GPTLanguageModel()
m = model.to(device)
print(f"Model has {sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
max_iters = 5000
eval_interval = 500

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")

BPE data shape: torch.Size([549002])
Model has 0.27M parameters
step 0: train loss 6.4885, val loss 6.4925
step 500: train loss 3.9575, val loss 4.1439
step 1000: train loss 3.6318, val loss 3.9112
step 1500: train loss 3.4218, val loss 3.7394
step 2000: train loss 3.2516, val loss 3.5997
step 2500: train loss 3.1536, val loss 3.5444
step 3000: train loss 3.0727, val loss 3.4840
step 3500: train loss 3.0217, val loss 3.4360
step 4000: train loss 2.9854, val loss 3.3946
step 4500: train loss 2.9391, val loss 3.3670

Final loss: 3.0969


In [16]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = m.generate(context, max_new_tokens=200)[0].tolist()
print(bpe_decode(generated))

 hatends.
My buard to have the coldness have come openits as holvices;
Net you forsand me there the pect to our knowwo me
kings. Come, make or sput a croptell younly fare,
But re, up the child shash flex eventlems,
Virs, soft will't, then's into we put on thee,
Hacting sofess you, let him in heevies,
But I promt you have least o'er: with oxament
Hadvers touts that for he was as'dd.

ISABELLA:
Aife made
O she do't


In [17]:
torch.save(model.state_dict(), 'shakespeare_gpt.pt')
print("Model saved!")

Model saved!
